In [1]:
# =============================================================================
# 1. Import Libraries
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, learning_curve
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [2]:
# 2. Load and Inspect Data
# =============================================================================
# Load the dataset (adjust path if needed)
df = pd.read_csv('car_price_dataset_cleaned.csv')

print("Dataset shape:", df.shape)
df.head()

# Check basic info
df.info()
df.describe()

# Check missing values
print("\nMissing values per column:")
print(df.isnull().sum())

Dataset shape: (9619, 15)
<class 'pandas.DataFrame'>
RangeIndex: 9619 entries, 0 to 9618
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           9619 non-null   str    
 1   Model           9619 non-null   str    
 2   Engine (cc)     9619 non-null   float64
 3   Gear            9619 non-null   str    
 4   Fuel Type       9619 non-null   str    
 5   Millage(KM)     9619 non-null   float64
 6   Town            9619 non-null   str    
 7   Leasing         9619 non-null   str    
 8   Condition       9619 non-null   str    
 9   AIR CONDITION   9619 non-null   str    
 10  POWER STEERING  9619 non-null   str    
 11  POWER MIRROR    9619 non-null   str    
 12  POWER WINDOW    9619 non-null   str    
 13  Price           9619 non-null   float64
 14  Car_Age         9619 non-null   int64  
dtypes: float64(3), int64(1), str(11)
memory usage: 1.6 MB

Missing values per column:
Brand             0
Model 

In [3]:
# =============================================================================
# 3. Data Preprocessing
# =============================================================================
# Separate target variable
target = 'Price'
X = df.drop(columns=[target])
y = df[target]

# Identify columns by type
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# Convert binary text columns ('Not_Available') to 0 for power features
# These columns: AIR CONDITION, POWER STEERING, POWER MIRROR, POWER WINDOW
# In the data, values are '1' or 'Not_Available'
binary_cols = ['AIR CONDITION', 'POWER STEERING', 'POWER MIRROR', 'POWER WINDOW']
for col in binary_cols:
    X[col] = X[col].map({'1': 1, 'Not_Available': 0}).fillna(0).astype(int)

# For 'Leasing' column, convert 'Ongoing Lease' to 1 and '0' to 0 (already numeric but object)
X['Leasing'] = X['Leasing'].map({'0': 0, 'Ongoing Lease': 1}).fillna(0).astype(int)

# For 'Condition' column, it's either 'USED' or 'NEW', convert to binary
X['Condition'] = X['Condition'].map({'USED': 0, 'NEW': 1}).fillna(0).astype(int)

# Now handle other categorical columns: Brand, Model, Gear, Fuel Type, Town
# We'll use OrdinalEncoder to convert them to integers (HistGradientBoosting can handle categoricals
# if we pass them as integer codes and specify categorical_features). We'll do that later.

# Separate numeric and categorical features
X_numeric = X[numeric_cols]
X_categorical = X[categorical_cols]

# Since HistGradientBoosting can handle missing values natively, we don't need to impute numeric.
# But we must encode categoricals.
# Use OrdinalEncoder (or LabelEncoder) for each column
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_cat_encoded = encoder.fit_transform(X_categorical)

# Convert to DataFrame for better handling
X_cat_encoded = pd.DataFrame(X_cat_encoded, columns=X_categorical.columns, index=X.index)

# Combine numeric and encoded categorical
X_prepared = pd.concat([X_numeric, X_cat_encoded], axis=1)

# Check for any remaining missing values
print("\nMissing values after preprocessing:")
print(X_prepared.isnull().sum().sum())

Numeric columns: ['Engine (cc)', 'Millage(KM)', 'Car_Age']
Categorical columns: ['Brand', 'Model', 'Gear', 'Fuel Type', 'Town', 'Leasing', 'Condition', 'AIR CONDITION', 'POWER STEERING', 'POWER MIRROR', 'POWER WINDOW']

Missing values after preprocessing:
0
